# Module 20 Lab: Agent Testing & Quality Assurance

Trong lab này, bạn sẽ xây dựng một **Test Framework hoàn chỉnh cho LoanBot** — từ unit tests đến chaos engineering.

## Lab Objectives
1. Implement deterministic và probabilistic tests
2. Xây dựng golden dataset evaluation pipeline
3. Viết property-based tests cho 6 core invariants
4. Build regression test baseline và drift detection
5. Implement chaos test suite với fault injection
6. Tích hợp thành CI/CD quality gate

In [ ]:
# Setup — Install dependencies
# pip install hypothesis pytest pytest-mock

import json, random, re, time, hashlib
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Callable
from enum import Enum
from unittest.mock import patch, MagicMock
from contextlib import contextmanager

print('✅ Dependencies ready')

## Section 1: Mock LoanBot (để test không cần Claude API)

In [ ]:
@dataclass
class LoanApplication:
    app_id: str
    credit_score: int
    dti_ratio: float       # debt-to-income
    employment_months: int
    loan_amount: int       # VNĐ
    blacklisted: bool = False

@dataclass
class LoanDecision:
    decision: str          # APPROVE / REJECT / REVIEW
    confidence: float      # 0.0 – 1.0
    explanation: str
    regulation_refs: List[str]
    factors: Dict[str, str]

class MockLoanBot:
    """Deterministic mock LoanBot for testing. Real LoanBot would call Claude API."""

    THRESHOLDS = {
        'credit_score_min': 620,
        'dti_max': 0.45,
        'employment_min': 6,   # months
        'credit_review_zone': (580, 620),   # borderline → REVIEW
    }

    def __init__(self, noise_level: float = 0.0):
        """noise_level: 0.0=deterministic, 0.1=10% chance of borderline variation."""
        self.noise_level = noise_level
        self._call_count = 0

    def evaluate(self, credit_score: int, dti_ratio: float = 0.35,
                 employment_months: int = 12, loan_amount: int = 300_000_000,
                 blacklisted: bool = False) -> dict:
        self._call_count += 1

        # INVARIANT: blacklisted always rejects
        if blacklisted:
            return self._make_result('REJECT', 0.99,
                f'Application {loan_amount:,} VNĐ rejected: applicant on blacklist.',
                ['TT39/2016', 'NĐ13/2023'], {'blacklist': 'FAIL'})

        fails = []
        factors = {}

        if credit_score < self.THRESHOLDS['credit_score_min']:
            review_min, review_max = self.THRESHOLDS['credit_review_zone']
            if credit_score >= review_min:
                factors['credit_score'] = 'BORDERLINE'
            else:
                factors['credit_score'] = 'FAIL'
                fails.append('credit_score')
        else:
            factors['credit_score'] = 'PASS'

        if dti_ratio > self.THRESHOLDS['dti_max']:
            factors['dti'] = 'FAIL'
            fails.append('dti')
        else:
            factors['dti'] = 'PASS'

        if employment_months < self.THRESHOLDS['employment_min']:
            factors['employment'] = 'FAIL'
            fails.append('employment')
        else:
            factors['employment'] = 'PASS'

        # Decision logic
        if len(fails) == 0 and 'BORDERLINE' not in factors.values():
            decision = 'APPROVE'
            confidence = 0.90 + (credit_score - 620) / 2300  # higher score → higher conf
        elif len(fails) >= 2:
            decision = 'REJECT'
            confidence = 0.92
        elif 'BORDERLINE' in factors.values() or len(fails) == 1:
            decision = 'REVIEW'
            confidence = 0.70
        else:
            decision = 'REJECT'
            confidence = 0.88

        confidence = min(0.99, max(0.50, confidence))

        explanation = (
            f"Loan application for {loan_amount:,} VNĐ evaluated. "
            f"Credit score {credit_score} ({'above' if credit_score >= 620 else 'below'} threshold 620). "
            f"DTI ratio {dti_ratio:.1%} ({'acceptable' if dti_ratio <= 0.45 else 'exceeds'} limit 45%). "
            f"Employment {employment_months} months ({'sufficient' if employment_months >= 6 else 'insufficient'})."
        )

        return self._make_result(decision, confidence, explanation,
                                  ['TT39/2016', 'EU-AI-Act-Art9'], factors)

    def _make_result(self, decision, confidence, explanation, refs, factors):
        return {
            'decision': decision,
            'confidence': confidence,
            'explanation': explanation,
            'regulation_refs': refs,
            'factors': factors
        }

# Test basic operation
bot = MockLoanBot()
result = bot.evaluate(credit_score=720, dti_ratio=0.32, employment_months=48)
print(f"Decision: {result['decision']} (conf: {result['confidence']:.2f})")
print(f"Explanation: {result['explanation'][:100]}...")
print(f"Factors: {result['factors']}")

## Section 2: Unit Tests — Deterministic & Probabilistic

In [ ]:
class UnitTestSuite:
    """Unit tests for LoanBot — deterministic and structural."""

    REQUIRED_FIELDS = {'decision', 'confidence', 'explanation', 'regulation_refs', 'factors'}
    VALID_DECISIONS = {'APPROVE', 'REJECT', 'REVIEW'}

    def __init__(self, loanbot):
        self.bot = loanbot
        self.results = []

    def _run(self, name: str, fn: Callable):
        try:
            fn()
            self.results.append((name, 'PASS', None))
            print(f'  ✅ {name}')
        except AssertionError as e:
            self.results.append((name, 'FAIL', str(e)))
            print(f'  ❌ {name}: {e}')

    def test_output_schema(self):
        """Output must always have required fields."""
        def run():
            result = self.bot.evaluate(credit_score=620, dti_ratio=0.35)
            missing = self.REQUIRED_FIELDS - result.keys()
            assert not missing, f'Missing fields: {missing}'
        self._run('Output schema completeness', run)

    def test_decision_valid_enum(self):
        """Decision must be one of APPROVE/REJECT/REVIEW."""
        def run():
            for score in [400, 580, 620, 720]:
                result = self.bot.evaluate(credit_score=score)
                assert result['decision'] in self.VALID_DECISIONS, \
                    f'score={score}: invalid decision {result["decision"]}'
        self._run('Decision valid enum', run)

    def test_confidence_range(self):
        """Confidence must be 0.0–1.0."""
        def run():
            for score in range(300, 851, 50):
                result = self.bot.evaluate(credit_score=score)
                conf = result['confidence']
                assert 0.0 <= conf <= 1.0, f'score={score}: confidence {conf} out of range'
        self._run('Confidence range [0, 1]', run)

    def test_blacklist_always_rejects(self):
        """Blacklisted applicant must always be REJECT regardless of score."""
        def run():
            for score in [400, 580, 720, 850]:
                result = self.bot.evaluate(credit_score=score, blacklisted=True)
                assert result['decision'] == 'REJECT', \
                    f'score={score}: blacklisted not rejected (got {result["decision"]})'
        self._run('Blacklist → always REJECT', run)

    def test_explanation_not_empty(self):
        """Explanation must be meaningful — not empty or minimal."""
        def run():
            result = self.bot.evaluate(credit_score=620, dti_ratio=0.40)
            exp = result['explanation']
            assert len(exp) >= 50, f'Explanation too short: {len(exp)} chars'
            assert 'credit' in exp.lower() or 'loan' in exp.lower(), \
                'Explanation does not mention credit/loan'
        self._run('Explanation quality (non-empty, relevant)', run)

    def test_regulation_refs_present(self):
        """Must always cite regulatory references — EU AI Act requirement."""
        def run():
            result = self.bot.evaluate(credit_score=700)
            refs = result['regulation_refs']
            assert len(refs) > 0, 'No regulation refs — EU AI Act Art.12 violation'
        self._run('Regulation refs present', run)

    def run_all(self):
        print('\n=== Unit Test Suite ===')
        self.test_output_schema()
        self.test_decision_valid_enum()
        self.test_confidence_range()
        self.test_blacklist_always_rejects()
        self.test_explanation_not_empty()
        self.test_regulation_refs_present()
        passed = sum(1 for _, s, _ in self.results if s == 'PASS')
        total = len(self.results)
        print(f'\nResult: {passed}/{total} passed ({passed/total:.0%})')
        return passed == total

bot = MockLoanBot()
suite = UnitTestSuite(bot)
all_pass = suite.run_all()

## Section 3: Golden Dataset Evaluation

In [ ]:
@dataclass
class GoldenCase:
    case_id: str
    credit_score: int
    dti_ratio: float
    employment_months: int
    loan_amount: int
    blacklisted: bool
    expert_decision: str   # APPROVE / REJECT / REVIEW
    expert_notes: str
    difficulty: str        # easy / borderline / edge_case

GOLDEN_DATASET: List[GoldenCase] = [
    # === Easy APPROVE cases ===
    GoldenCase('GD-001', 720, 0.32, 48, 500_000_000, False, 'APPROVE', 'Strong profile across all metrics', 'easy'),
    GoldenCase('GD-002', 750, 0.28, 60, 300_000_000, False, 'APPROVE', 'Excellent profile', 'easy'),
    GoldenCase('GD-003', 680, 0.38, 24, 400_000_000, False, 'APPROVE', 'Good profile, all thresholds clear', 'easy'),
    GoldenCase('GD-004', 710, 0.30, 36, 250_000_000, False, 'APPROVE', 'Conservative loan amount for income', 'easy'),
    GoldenCase('GD-005', 690, 0.35, 18, 200_000_000, False, 'APPROVE', 'Passes all criteria', 'easy'),
    # === Easy REJECT cases ===
    GoldenCase('GD-006', 400, 0.75, 2,  200_000_000, False, 'REJECT',  'All metrics fail significantly', 'easy'),
    GoldenCase('GD-007', 450, 0.65, 3,  500_000_000, False, 'REJECT',  'High risk profile', 'easy'),
    GoldenCase('GD-008', 720, 0.32, 48, 300_000_000, True,  'REJECT',  'Blacklisted regardless of score', 'easy'),
    GoldenCase('GD-009', 380, 0.80, 1,  100_000_000, False, 'REJECT',  'Severely failing all metrics', 'easy'),
    GoldenCase('GD-010', 500, 0.60, 4,  400_000_000, False, 'REJECT',  'Multiple criteria fail', 'easy'),
    # === Borderline cases ===
    GoldenCase('GD-011', 625, 0.44, 7,  300_000_000, False, 'APPROVE', 'Just above all thresholds', 'borderline'),
    GoldenCase('GD-012', 618, 0.43, 8,  250_000_000, False, 'REVIEW',  'Credit score borderline', 'borderline'),
    GoldenCase('GD-013', 630, 0.46, 6,  350_000_000, False, 'REVIEW',  'DTI marginal', 'borderline'),
    GoldenCase('GD-014', 645, 0.44, 5,  280_000_000, False, 'REVIEW',  'Employment borderline', 'borderline'),
    GoldenCase('GD-015', 622, 0.45, 6,  300_000_000, False, 'APPROVE', 'All exactly at threshold — approve', 'borderline'),
    GoldenCase('GD-016', 619, 0.46, 5,  400_000_000, False, 'REJECT',  'Two criteria just below threshold', 'borderline'),
    GoldenCase('GD-017', 580, 0.44, 10, 200_000_000, False, 'REVIEW',  'Low credit but other metrics OK', 'borderline'),
    GoldenCase('GD-018', 640, 0.44, 6,  500_000_000, False, 'REVIEW',  'High amount, borderline profile', 'borderline'),
    # === Edge cases ===
    GoldenCase('GD-019', 620, 0.45, 6,  300_000_000, False, 'APPROVE', 'Exactly at all thresholds', 'edge_case'),
    GoldenCase('GD-020', 300, 0.35, 12, 100_000_000, False, 'REJECT',  'Minimum possible credit score', 'edge_case'),
]

def evaluate_golden_dataset(bot, dataset: List[GoldenCase]) -> dict:
    total = len(dataset)
    correct = 0
    errors = []
    by_difficulty = {}

    for case in dataset:
        result = bot.evaluate(
            credit_score=case.credit_score, dti_ratio=case.dti_ratio,
            employment_months=case.employment_months, loan_amount=case.loan_amount,
            blacklisted=case.blacklisted
        )
        predicted = result['decision']
        is_correct = predicted == case.expert_decision
        if is_correct:
            correct += 1
        else:
            errors.append({'id': case.case_id, 'expected': case.expert_decision,
                           'got': predicted, 'difficulty': case.difficulty,
                           'notes': case.expert_notes})
        d = case.difficulty
        if d not in by_difficulty:
            by_difficulty[d] = {'correct': 0, 'total': 0}
        by_difficulty[d]['total'] += 1
        if is_correct:
            by_difficulty[d]['correct'] += 1

    accuracy = correct / total
    by_diff_acc = {
        k: v['correct'] / v['total']
        for k, v in by_difficulty.items()
    }
    # Thresholds: easy ≥ 95%, borderline ≥ 85%, edge_case ≥ 70%
    thresholds = {'easy': 0.95, 'borderline': 0.85, 'edge_case': 0.70}
    tier_pass = {
        k: by_diff_acc.get(k, 0) >= thresholds.get(k, 0.90)
        for k in thresholds
    }
    return {
        'total': total, 'correct': correct, 'accuracy': accuracy,
        'overall_pass': accuracy >= 0.90 and all(tier_pass.values()),
        'by_difficulty': by_diff_acc, 'tier_pass': tier_pass, 'errors': errors
    }

bot = MockLoanBot()
report = evaluate_golden_dataset(bot, GOLDEN_DATASET)

print(f"=== Golden Dataset Evaluation ===")
print(f"Overall: {report['correct']}/{report['total']} ({report['accuracy']:.1%}) — {'✅ PASS' if report['overall_pass'] else '❌ FAIL'}")
print(f"\nBy difficulty:")
for diff, acc in report['by_difficulty'].items():
    status = '✅' if report['tier_pass'].get(diff, False) else '❌'
    print(f"  {status} {diff}: {acc:.1%}")
if report['errors']:
    print(f"\nErrors ({len(report['errors'])}):")
    for e in report['errors']:
        print(f"  {e['id']} [{e['difficulty']}]: expected {e['expected']}, got {e['got']} — {e['notes']}")

## Section 4: Property-Based Tests — 6 Invariants

In [ ]:
class PropertyTestSuite:
    """Property-based tests — verify invariants across many random inputs."""

    DECISION_RANK = {'APPROVE': 2, 'REVIEW': 1, 'REJECT': 0}

    def __init__(self, bot, n_random: int = 100):
        self.bot = bot
        self.n_random = n_random
        self.results = []

    def _run(self, name: str, fn: Callable):
        try:
            fn()
            self.results.append((name, 'PASS', None))
            print(f'  ✅ {name}')
        except AssertionError as e:
            self.results.append((name, 'FAIL', str(e)))
            print(f'  ❌ {name}: {e}')

    def prop_schema_always_valid(self):
        """INVARIANT 1: Output schema always valid for any input."""
        def run():
            for i in range(self.n_random):
                score = random.randint(300, 850)
                dti   = round(random.uniform(0.0, 1.0), 2)
                emp   = random.randint(0, 120)
                result = self.bot.evaluate(credit_score=score, dti_ratio=dti, employment_months=emp)
                assert result['decision'] in {'APPROVE', 'REJECT', 'REVIEW'}, \
                    f'iter {i}: invalid decision {result["decision"]}'
                assert 0.0 <= result['confidence'] <= 1.0, \
                    f'iter {i}: confidence {result["confidence"]} out of range'
                assert isinstance(result['explanation'], str) and len(result['explanation']) > 0, \
                    f'iter {i}: empty explanation'
        self._run(f'P1: Schema always valid ({self.n_random} random inputs)', run)

    def prop_blacklist_always_rejects(self):
        """INVARIANT 2: Blacklisted always → REJECT, no matter the score."""
        def run():
            for score in range(300, 851, 25):
                result = self.bot.evaluate(credit_score=score, blacklisted=True)
                assert result['decision'] == 'REJECT', \
                    f'score={score}: blacklisted got {result["decision"]} (expected REJECT)'
        self._run('P2: Blacklist → always REJECT', run)

    def prop_monotonicity_credit_score(self):
        """INVARIANT 3: Higher credit score → decision rank ≥ lower score (same other factors)."""
        def run():
            dti, emp = 0.38, 12
            violations = []
            scores = range(300, 851, 25)
            prev_rank, prev_score = -1, None
            for score in scores:
                r = self.bot.evaluate(credit_score=score, dti_ratio=dti, employment_months=emp)
                rank = self.DECISION_RANK[r['decision']]
                if prev_rank >= 0 and rank < prev_rank:
                    violations.append(f'score {prev_score}→{score}: {list(self.DECISION_RANK.keys())[2-prev_rank]}→{r["decision"]}')
                prev_rank, prev_score = rank, score
            assert not violations, f'Monotonicity violations: {violations}'
        self._run('P3: Credit score monotonicity', run)

    def prop_no_hallucinated_amounts(self):
        """INVARIANT 4: Explanation must not mention loan amounts not in input."""
        def run():
            test_amounts = [100_000_000, 300_000_000, 500_000_000, 1_000_000_000]
            for amount in test_amounts:
                result = self.bot.evaluate(credit_score=680, dti_ratio=0.35, loan_amount=amount)
                exp = result['explanation']
                # Extract large numbers from explanation
                numbers = [int(n.replace(',', '')) for n in re.findall(r'\d[\d,]{5,}', exp)]
                for num in numbers:
                    if num > 1_000_000:  # significant amounts
                        assert abs(num - amount) < amount * 0.01, \
                            f'amount={amount:,}: hallucinated {num:,} in explanation'
        self._run('P4: No hallucinated loan amounts', run)

    def prop_regulation_refs_always_present(self):
        """INVARIANT 5: Regulation refs must always be non-empty."""
        def run():
            for i in range(20):
                score = random.randint(300, 850)
                result = self.bot.evaluate(credit_score=score)
                assert len(result['regulation_refs']) > 0, \
                    f'iter {i}: no regulation refs (EU AI Act Art.12 violation)'
        self._run('P5: Regulation refs always present', run)

    def prop_high_confidence_on_extremes(self):
        """INVARIANT 6: Clear approve/reject cases must have high confidence."""
        def run():
            # Clear APPROVE case
            r = self.bot.evaluate(credit_score=800, dti_ratio=0.20, employment_months=60)
            assert r['decision'] == 'APPROVE' and r['confidence'] >= 0.85, \
                f'Strong approve: conf={r["confidence"]:.2f} (expected ≥0.85)'
            # Clear REJECT case
            r = self.bot.evaluate(credit_score=350, dti_ratio=0.80, employment_months=1)
            assert r['decision'] == 'REJECT' and r['confidence'] >= 0.85, \
                f'Strong reject: conf={r["confidence"]:.2f} (expected ≥0.85)'
        self._run('P6: High confidence on clear cases', run)

    def run_all(self):
        print('\n=== Property-Based Test Suite ===')
        self.prop_schema_always_valid()
        self.prop_blacklist_always_rejects()
        self.prop_monotonicity_credit_score()
        self.prop_no_hallucinated_amounts()
        self.prop_regulation_refs_always_present()
        self.prop_high_confidence_on_extremes()
        passed = sum(1 for _, s, _ in self.results if s == 'PASS')
        total = len(self.results)
        print(f'\nResult: {passed}/{total} passed ({passed/total:.0%})')
        return passed == total

bot = MockLoanBot()
prop_suite = PropertyTestSuite(bot, n_random=100)
prop_suite.run_all()

## Section 5: Regression Testing — Baseline & Drift Detection

In [ ]:
import hashlib

class RegressionTestFramework:
    """Snapshot-based regression testing — detect decision drift after changes."""

    REGRESSION_CASES = [
        {'id': 'REG-001', 'credit_score': 720, 'dti_ratio': 0.32, 'employment_months': 48, 'loan_amount': 500_000_000},
        {'id': 'REG-002', 'credit_score': 580, 'dti_ratio': 0.58, 'employment_months': 10, 'loan_amount': 300_000_000},
        {'id': 'REG-003', 'credit_score': 645, 'dti_ratio': 0.38, 'employment_months': 14, 'loan_amount': 400_000_000},
        {'id': 'REG-004', 'credit_score': 400, 'dti_ratio': 0.75, 'employment_months': 2,  'loan_amount': 200_000_000},
        {'id': 'REG-005', 'credit_score': 625, 'dti_ratio': 0.44, 'employment_months': 7,  'loan_amount': 300_000_000},
        {'id': 'REG-006', 'credit_score': 618, 'dti_ratio': 0.43, 'employment_months': 8,  'loan_amount': 250_000_000},
        {'id': 'REG-007', 'credit_score': 700, 'dti_ratio': 0.30, 'employment_months': 24, 'loan_amount': 450_000_000},
        {'id': 'REG-008', 'credit_score': 550, 'dti_ratio': 0.50, 'employment_months': 5,  'loan_amount': 300_000_000},
    ]

    def __init__(self):
        self.baseline: Dict[str, str] = {}  # case_id → decision
        self.baseline_hash: str = ''

    def capture_baseline(self, bot) -> None:
        """Capture baseline decisions from current bot version."""
        self.baseline = {}
        for case in self.REGRESSION_CASES:
            case_id = case['id']
            result = bot.evaluate(**{k: v for k, v in case.items() if k != 'id'})
            self.baseline[case_id] = result['decision']
        self.baseline_hash = hashlib.md5(
            json.dumps(self.baseline, sort_keys=True).encode()
        ).hexdigest()[:8]
        print(f'Baseline captured: {len(self.baseline)} cases, hash={self.baseline_hash}')
        for cid, dec in self.baseline.items():
            print(f'  {cid}: {dec}')

    def check_regression(self, new_bot, threshold: float = 0.02) -> dict:
        """Compare new bot against baseline. Block if drift > threshold."""
        if not self.baseline:
            raise RuntimeError('No baseline captured. Call capture_baseline() first.')

        changes = []
        for case in self.REGRESSION_CASES:
            case_id = case['id']
            result = new_bot.evaluate(**{k: v for k, v in case.items() if k != 'id'})
            new_dec = result['decision']
            old_dec = self.baseline[case_id]
            if new_dec != old_dec:
                changes.append({'id': case_id, 'from': old_dec, 'to': new_dec})

        total = len(self.REGRESSION_CASES)
        drift = len(changes) / total
        return {
            'changes': changes,
            'drift': drift,
            'pass': drift <= threshold,
            'verdict': '✅ PASS' if drift <= threshold else f'❌ FAIL (drift {drift:.1%} > threshold {threshold:.0%})'
        }

# Demo: capture baseline and then test a 'new' version
framework = RegressionTestFramework()
v1_bot = MockLoanBot()
framework.capture_baseline(v1_bot)

print('\n--- Testing v1 against own baseline (should be 0% drift) ---')
report = framework.check_regression(v1_bot)
print(f"Drift: {report['drift']:.1%}, Changes: {len(report['changes'])}, {report['verdict']}")

# Simulate a 'buggy' v2 that changes some decisions
class BuggyLoanBot(MockLoanBot):
    def evaluate(self, **kwargs):
        result = super().evaluate(**kwargs)
        # Bug: flips REVIEW → APPROVE for scores 615-640
        if result['decision'] == 'REVIEW' and 615 <= kwargs.get('credit_score', 0) <= 640:
            result['decision'] = 'APPROVE'
        return result

print('\n--- Testing BuggyBot v2 against v1 baseline ---')
v2_buggy = BuggyLoanBot()
report2 = framework.check_regression(v2_buggy)
print(f"Drift: {report2['drift']:.1%}, Changes: {len(report2['changes'])}, {report2['verdict']}")
for change in report2['changes']:
    print(f"  {change['id']}: {change['from']} → {change['to']}")

## Section 6: Chaos Test Suite

In [ ]:
class ChaosTestSuite:
    """Fault injection tests — verify LoanBot resilience."""

    def __init__(self, bot):
        self.bot = bot
        self.results = []

    def _run(self, name: str, fn: Callable):
        try:
            fn()
            self.results.append((name, 'PASS', None))
            print(f'  ✅ {name}')
        except AssertionError as e:
            self.results.append((name, 'FAIL', str(e)))
            print(f'  ❌ {name}: {e}')

    def test_resilience_under_timeout(self):
        """Simulate 30% API timeout rate — expect ≥90% success with retry."""
        class FlakyBot:
            def __init__(self, real_bot, fail_rate=0.3):
                self.real_bot = real_bot
                self.fail_rate = fail_rate
                self.attempt_counts = []

            def evaluate(self, **kwargs):
                # Simulate retry logic (up to 3 attempts)
                for attempt in range(3):
                    if random.random() >= self.fail_rate:
                        self.attempt_counts.append(attempt + 1)
                        return self.real_bot.evaluate(**kwargs)
                raise TimeoutError('All 3 attempts timed out')

        def run():
            flaky = FlakyBot(self.bot, fail_rate=0.3)
            n_tests, successes = 50, 0
            random.seed(42)  # reproducible
            for i in range(n_tests):
                try:
                    result = flaky.evaluate(credit_score=620+i%100, dti_ratio=0.35)
                    if result['decision'] in {'APPROVE', 'REJECT', 'REVIEW'}:
                        successes += 1
                except TimeoutError:
                    pass
            success_rate = successes / n_tests
            avg_attempts = sum(flaky.attempt_counts) / len(flaky.attempt_counts) if flaky.attempt_counts else 0
            assert success_rate >= 0.90, f'Resilience: {success_rate:.0%} (need ≥90%) — avg {avg_attempts:.1f} attempts'
            print(f'     success={success_rate:.0%}, avg_attempts={avg_attempts:.1f}')
        self._run('Resilience under 30% timeout rate', run)

    def test_graceful_degradation_on_invalid_input(self):
        """Invalid inputs should not crash — return safe fallback."""
        def run():
            edge_inputs = [
                {'credit_score': 300, 'dti_ratio': 0.0, 'employment_months': 0},   # minimums
                {'credit_score': 850, 'dti_ratio': 1.0, 'employment_months': 120}, # maximums
                {'credit_score': 620, 'dti_ratio': 0.35, 'loan_amount': 0},        # zero amount
                {'credit_score': 620, 'dti_ratio': 0.35, 'loan_amount': 10_000_000_000},  # huge amount
            ]
            for inp in edge_inputs:
                try:
                    result = self.bot.evaluate(**inp)
                    assert result is not None
                    assert result['decision'] in {'APPROVE', 'REJECT', 'REVIEW'}
                except Exception as e:
                    assert False, f'Crashed on input {inp}: {e}'
        self._run('Graceful degradation on edge inputs', run)

    def test_concurrent_evaluation(self):
        """LoanBot should handle concurrent requests without state corruption."""
        import threading
        def run():
            results = []
            errors = []

            def evaluate_and_collect(score):
                try:
                    r = self.bot.evaluate(credit_score=score, dti_ratio=0.35)
                    results.append(r['decision'])
                except Exception as e:
                    errors.append(str(e))

            threads = [threading.Thread(target=evaluate_and_collect, args=(500+i*20,))
                       for i in range(10)]
            [t.start() for t in threads]
            [t.join() for t in threads]

            assert not errors, f'Concurrent errors: {errors}'
            assert len(results) == 10
            assert all(r in {'APPROVE', 'REJECT', 'REVIEW'} for r in results)
        self._run('Concurrent evaluation (10 threads)', run)

    def run_all(self):
        print('\n=== Chaos Test Suite ===')
        self.test_resilience_under_timeout()
        self.test_graceful_degradation_on_invalid_input()
        self.test_concurrent_evaluation()
        passed = sum(1 for _, s, _ in self.results if s == 'PASS')
        total = len(self.results)
        print(f'\nResult: {passed}/{total} passed ({passed/total:.0%})')
        return passed == total

bot = MockLoanBot()
chaos = ChaosTestSuite(bot)
chaos.run_all()

## Section 7: CI/CD Quality Gate — Integration

In [ ]:
class QualityGate:
    """Full CI/CD quality gate — runs all test suites, blocks if any fail."""

    def __init__(self, bot, regression_framework: RegressionTestFramework):
        self.bot = bot
        self.regression = regression_framework
        self.gate_results = {}

    def run(self) -> bool:
        print('=' * 55)
        print('  LoanBot CI/CD Quality Gate')
        print('=' * 55)

        # Stage 1: Unit Tests
        print('\n[STAGE 1] Unit Tests...')
        unit = UnitTestSuite(self.bot)
        self.gate_results['unit'] = unit.run_all()

        # Stage 2: Property Tests
        print('\n[STAGE 2] Property Tests...')
        prop = PropertyTestSuite(self.bot, n_random=50)
        self.gate_results['property'] = prop.run_all()

        # Stage 3: Golden Dataset
        print('\n[STAGE 3] Golden Dataset...')
        gd_report = evaluate_golden_dataset(self.bot, GOLDEN_DATASET)
        self.gate_results['golden'] = gd_report['overall_pass']
        print(f'  Accuracy: {gd_report["accuracy"]:.1%} — {"✅ PASS" if gd_report["overall_pass"] else "❌ FAIL"}')

        # Stage 4: Regression
        print('\n[STAGE 4] Regression Tests...')
        reg_report = self.regression.check_regression(self.bot)
        self.gate_results['regression'] = reg_report['pass']
        print(f'  {reg_report["verdict"]}')

        # Stage 5: Chaos Tests
        print('\n[STAGE 5] Chaos Tests...')
        chaos = ChaosTestSuite(self.bot)
        self.gate_results['chaos'] = chaos.run_all()

        # Final verdict
        all_pass = all(self.gate_results.values())
        print('\n' + '=' * 55)
        print('  QUALITY GATE SUMMARY')
        print('=' * 55)
        for stage, passed in self.gate_results.items():
            print(f'  {"✅" if passed else "❌"} {stage.capitalize()}')
        print('=' * 55)
        verdict = '✅ ALL STAGES PASSED — DEPLOYMENT APPROVED' if all_pass else '❌ QUALITY GATE FAILED — BLOCK DEPLOYMENT'
        print(f'  {verdict}')
        print('=' * 55)
        return all_pass

# Capture baseline first
reg_fw = RegressionTestFramework()
reg_fw.capture_baseline(MockLoanBot())

# Run full quality gate
gate = QualityGate(MockLoanBot(), reg_fw)
gate.run()

## Section 8: Practice — Build Your Own Test

**Bài tập:** FinTech Corp muốn thêm test mới: verify rằng khi `loan_amount > 800,000,000 VNĐ`, LoanBot phải luôn route về REVIEW (không auto-APPROVE) để có human oversight cho large loans.

Implement invariant này trong `PropertyTestSuite.prop_large_loans_require_review()` và chạy nó.

In [ ]:
# TODO: Implement prop_large_loans_require_review
# Hint: loop through large amounts (800M, 900M, 1B, 1.5B)
#       with strong profiles (score=720, dti=0.30, emp=24)
#       and assert decision != 'APPROVE'

class ExtendedPropertyTests(PropertyTestSuite):
    def prop_large_loans_require_review(self):
        """INVARIANT: Large loans (>800M VNĐ) must not be auto-approved."""
        def run():
            # TODO: implement here
            pass
        self._run('P7: Large loans → not auto-APPROVE', run)

# SOLUTION (uncomment to check):
# class ExtendedPropertyTests(PropertyTestSuite):
#     def prop_large_loans_require_review(self):
#         def run():
#             large_amounts = [800_000_001, 900_000_000, 1_000_000_000, 1_500_000_000]
#             for amount in large_amounts:
#                 result = self.bot.evaluate(
#                     credit_score=720, dti_ratio=0.30, employment_months=24,
#                     loan_amount=amount
#                 )
#                 # Note: MockLoanBot doesn't implement this rule yet — test will FAIL
#                 # That's the point: TDD — write failing test first, then implement
#                 assert result['decision'] != 'APPROVE', \
#                     f'amount={amount:,}: large loan should not auto-approve (got APPROVE)'
#         self._run('P7: Large loans → not auto-APPROVE', run)

bot = MockLoanBot()
ext = ExtendedPropertyTests(bot, n_random=20)
ext.prop_large_loans_require_review()
print('\n💡 Note: Test likely FAIL — that is TDD: write the test first, then implement the rule in MockLoanBot!')